# Restoring Tone Marks on Vietnamese Place Names Built from Unseen Syllables

Three progressive submissions:
1. **Copy** the folded input unchanged (sanity floor, scores 0).
2. A **character n-gram** context rule (shown for reference; a hand-coded rule
   is **not** a valid submission under the task Rules, and on this split it
   scores below the trained model, because every test name contains a
   syllable absent from training).
3. The intended solution: a **character Transformer trained on the GPU** that
   restores the tone and vowel diacritics, learning the orthography from the
   training pairs only. Early stopping uses the shipped `grade()` on a
   held-out slice of the training rows. This cell uses the GPU.

Each cell writes `working/submission.csv` with columns `id,name`.


In [ ]:
import os, csv, re
from pathlib import Path

# Robust data-path discovery (never hardcode one location).
DATA = next(p for p in (Path("/data"), Path("dataset/public"),
                        Path("/kaggle/input/restoring-vietnamese-diacritics/public"))
            if (p / "test.csv").exists())
WORK = Path("working"); WORK.mkdir(exist_ok=True)

def read(p):
    with open(p, encoding="utf-8", newline="") as fh:
        return list(csv.DictReader(fh))

train = read(DATA / "train.csv")
test  = read(DATA / "test.csv")
print("train", len(train), "test", len(test))

def write_submission(pred_map):
    with open(WORK / "submission.csv", "w", encoding="utf-8", newline="") as fh:
        w = csv.writer(fh); w.writerow(["id", "name"])
        for r in test:
            w.writerow([r["id"], pred_map[r["id"]]])


In [ ]:
# Submission 1: copy baseline (scores 0 under changed-position F1)
write_submission({r["id"]: r["folded"] for r in test})
print("wrote copy baseline")


In [ ]:
# Submission 2: character n-gram context rule (reference only; NOT a valid
# submission -- no trained neural parameters). Shown to illustrate the baseline.
from collections import defaultdict, Counter
W = 2  # 5-gram window
tab = defaultdict(Counter)
for r in train:                      # train only
    a, t = r["folded"], r["name"]
    for i in range(len(a)):
        ctx = tuple(a[i+d] if 0 <= i+d < len(a) else "#" for d in range(-W, W+1))
        tab[ctx][t[i]] += 1
def ngram_pred(a):
    o = []
    for i in range(len(a)):
        ctx = tuple(a[i+d] if 0 <= i+d < len(a) else "#" for d in range(-W, W+1))
        o.append(tab[ctx].most_common(1)[0][0] if tab[ctx] else a[i])
    return "".join(o)
write_submission({r["id"]: ngram_pred(r["folded"]) for r in test})
print("wrote n-gram rule baseline")


In [ ]:
# Submission 3 (intended): character Transformer trained on the GPU, reading
# the folded name. Vocabulary is built from the TRAINING data only. A random
# 10% of train.csv is held out as a dev set for early stopping with the
# shipped grade() -- no test leakage, every dev row comes from train.csv.
import torch, torch.nn as nn, random
from grade import grade
random.seed(0); torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

sh = train[:]; random.Random(1).shuffle(sh)
n_dev = max(200, round(len(sh) * 0.1))
dev_rows, fit_rows = sh[:n_dev], sh[n_dev:]
dev_answers = [{"id": r["id"], "name": r["name"], "folded_src": r["folded"],
                "province_src": r["province"], "district_src": r["district"],
                "neighbors_src": r["neighbors"]} for r in dev_rows]

in_c  = sorted({c for r in fit_rows for c in r["folded"]})   # train only
out_c = sorted({c for r in fit_rows for c in r["name"]})     # train only
si = {c: i+1 for i, c in enumerate(in_c)}
oi = {c: i for i, c in enumerate(out_c)}
ML = max(len(r["folded"]) for r in fit_rows) + 2

def enc(a): return [si.get(c, 0) for c in a]
def pad(b):
    L = max(len(x) for x in b)
    return torch.tensor([x + [0]*(L-len(x)) for x in b], device=device)

class CharTagger(nn.Module):
    def __init__(self, D=256, L=4, H=8, dropout=0.15):
        super().__init__()
        self.e = nn.Embedding(len(in_c)+1, D, padding_idx=0)
        self.p = nn.Embedding(ML+2, D)
        lay = nn.TransformerEncoderLayer(D, H, D*4, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(lay, L, enable_nested_tensor=False)
        self.h = nn.Linear(D, len(out_c))
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.h(self.enc(self.e(x) + self.p(pos), src_key_padding_mask=(x == 0)))

MAX_EPOCHS, PATIENCE = 40, 6   # minutes on an A10G
model = CharTagger().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
lf = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.05)
ex = [(enc(r["folded"]), [oi.get(c, 0) for c in r["name"]]) for r in fit_rows]

def predict(rows):
    model.eval(); out = {}
    with torch.no_grad():
        for i in range(0, len(rows), 512):
            b = rows[i:i+512]
            lg = model(pad([enc(r["folded"]) for r in b])).argmax(-1).cpu()
            for k, r in enumerate(b):
                out[r["id"]] = "".join(out_c[lg[k, j].item()] for j in range(len(r["folded"])))
    model.train(); return out

def dev_score():
    p = predict(dev_rows)
    return grade([{"id": i, "name": n} for i, n in p.items()], dev_answers)

best, best_state, best_ep, bad = -1.0, None, 0, 0
for ep in range(MAX_EPOCHS):
    random.shuffle(ex)
    for i in range(0, len(ex), 256):
        ch = ex[i:i+256]
        x = pad([c[0] for c in ch])
        y = pad([c[1] for c in ch]).masked_fill(x == 0, -100)
        loss = lf(model(x).reshape(-1, len(out_c)), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    d = dev_score()
    if d > best:
        best, best_ep, bad = d, ep + 1, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
    if bad >= PATIENCE:
        break
print(f"best dev epoch {best_ep} (dev score {best:.4f})")

model.load_state_dict(best_state)
write_submission(predict(test))
print("wrote trained GPU submission")